# 01 — Sanity Check: tokenizer wrappers on Korean

**Goal**: confirm each provider's `Tokenizer` wrapper produces token counts in the expected range for Korean text *before* we trust any later experiment. New wrappers are added one at a time; PASS here is the gate to add the next.

**Per-provider expected TPC range** (calibrated on n=10 as each provider is added).

| Provider | Korean | English | Notes |
|---|---|---|---|
| GPT-4o (`o200k_base`) | 0.50 – 0.80 | 0.15 – 0.30 | observed 0.664 / 0.196 |
| Claude Sonnet 4.5 | 0.90 – 1.30 | 0.20 – 0.35 | observed 1.074 / 0.213 (PENALTY) |
| Gemini 2.5 Flash | 0.50 – 0.80 | 0.15 – 0.30 | observed 0.648 / 0.200 (GPT-like) |
| Solar 10.7B (Upstage) | 0.90 – 1.30 | 0.20 – 0.35 | observed 1.139 / 0.217 (PENALTY) |
| EXAONE 3.5 7.8B (LG) | 0.30 – 1.20 *(loose)* | 0.15 – 0.45 *(loose)* | tighten after first measurement |
| Llama / Qwen (later) | TBD | TBD | |

**Phase 1 surprise hypotheses** (we are not testing them here yet, but later runs will):

- **H1** — frontier models meaningfully improved Korean efficiency  
- **H2** — Korean-specialized models win on TPC but lose on ECPC after pricing  
- **H3** — medical / 한자-mixed text has a far higher penalty than everyday text

**Working hypothesis after 4 measurements (cautious — n=10 is informational only)**:

Four tokenizers cluster into two groups by KPR (= TPC_korean / TPC_english):
- **Efficient cluster** (KPR ≈ 3.3×): GPT-4o, Gemini 2.5 Flash — explicit Korean BPE merges
- **Penalty cluster** (KPR ≈ 5.1×): Claude Sonnet 4.5, Solar 10.7B — byte-level fallback for Hangul

Korean-targeted pretraining (Solar's depth-up-scaling on Mistral) does **not** bridge this gap — Solar inherits Mistral's English-centric tokenizer and pays the same 1.5× asymmetry penalty as Claude. The tokenizer's vocab coverage *appears to* dominate pretraining target language for cost efficiency. **Verify on full Phase 1 corpus before treating this as paper-citable.**

**EXAONE is the direct test of the working hypothesis.** EXAONE has explicit Korean vocab extension on top of a Llama-style base. If KPR/GPT < 1.3× (efficient or advantage), the working hypothesis is supported. If EXAONE also shows penalty, the hypothesis breaks and we investigate other factors.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the in-tree package importable without installing it.
# Once you `uv sync` and `pip install -e .` you can drop these two lines.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from korean_llm_cost.tokenizers.openai_tok import OpenAITokenizer

tok = OpenAITokenizer("gpt-4o")
print(tok)

<OpenAITokenizer gpt-4o@2024-11-20>


In [2]:
# 10 Korean samples across categories we plan to measure in Phase 1,
# plus a few harder cases (한자 mixed, code mixed, exclamation) to see
# the per-sample variance even at this tiny scale.
korean_samples: list[tuple[str, str]] = [
    ("conv",       "오늘 점심 뭐 먹을까?"),
    ("conv",       "내일 회의 시간 좀 알려줘."),
    ("news",       "한국은행은 기준금리를 동결하기로 결정했다."),
    ("news",       "정부는 내년도 예산안을 국회에 제출하였다."),
    ("medical",    "환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다."),
    ("medical",    "고혈압성 좌심실 비대 소견이 관찰되었다."),
    ("academic",   "본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다."),
    ("code_mixed", "리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다."),
    ("hanja",      "大韓民國의 헌법 제1조는 민주공화국임을 천명한다."),
    ("short",      "와 진짜 대박이네요!"),
]

english_baseline: list[str] = [
    "What should we eat for lunch today?",
    "Let me know the meeting time tomorrow.",
    "The Bank of Korea decided to freeze the benchmark interest rate.",
    "This study quantitatively analyzes the efficiency of Korean tokenizers.",
    "Wow, that's really amazing!",
]

len(korean_samples), len(english_baseline)

(10, 5)

In [3]:
def tpc(text: str) -> float:
    """Tokens per character. Aggregate over many texts via summed counts,
    not averaged TPCs — averaging per-text TPC over-weights short texts."""
    return tok.count(text) / len(text) if text else 0.0


print(f"{'category':<12} {'chars':>5} {'toks':>5} {'tpc':>6}  text")
print("-" * 80)
for category, text in korean_samples:
    n_chars = len(text)
    n_toks = tok.count(text)
    print(f"{category:<12} {n_chars:>5} {n_toks:>5} {n_toks/n_chars:>6.3f}  {text}")

category     chars  toks    tpc  text
--------------------------------------------------------------------------------
conv            12     8  0.667  오늘 점심 뭐 먹을까?
conv            15     9  0.600  내일 회의 시간 좀 알려줘.
news            23    13  0.565  한국은행은 기준금리를 동결하기로 결정했다.
news            23    13  0.565  정부는 내년도 예산안을 국회에 제출하였다.
medical         40    30  0.750  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22    17  0.773  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33    20  0.606  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38    22  0.579  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27    22  0.815  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11     8  0.727  와 진짜 대박이네요!


In [4]:
# Aggregate TPC: total tokens / total chars (NOT mean of per-text TPCs).
ko_chars = sum(len(t) for _, t in korean_samples)
ko_toks = sum(tok.count(t) for _, t in korean_samples)
en_chars = sum(len(t) for t in english_baseline)
en_toks = sum(tok.count(t) for t in english_baseline)

ko_tpc = ko_toks / ko_chars
en_tpc = en_toks / en_chars
kpr_gpt = ko_tpc / en_tpc  # rough; real KPR needs aligned parallel data

print(f"GPT-4o Korean : {ko_toks:>4} tokens / {ko_chars:>4} chars = TPC {ko_tpc:.3f}")
print(f"GPT-4o English: {en_toks:>4} tokens / {en_chars:>4} chars = TPC {en_tpc:.3f}")
print(f"GPT-4o KPR (ko / en) = {kpr_gpt:.2f}×")

# Pass / fail against GPT-4o-specific ranges (each provider gets its own band).
ok_ko = 0.5 <= ko_tpc <= 0.8
ok_en = 0.15 <= en_tpc <= 0.30
print()
print(f"GPT-4o Korean  TPC in [0.50, 0.80]? ", "PASS" if ok_ko else f"FAIL ({ko_tpc:.3f})")
print(f"GPT-4o English TPC in [0.15, 0.30]? ", "PASS" if ok_en else f"FAIL ({en_tpc:.3f})")

GPT-4o Korean :  162 tokens /  244 chars = TPC 0.664
GPT-4o English:   46 tokens /  235 chars = TPC 0.196
GPT-4o KPR (ko / en) = 3.39×

GPT-4o Korean  TPC in [0.50, 0.80]?  PASS
GPT-4o English TPC in [0.15, 0.30]?  PASS


## Anthropic (Claude) check — including English baseline

Same 10 Korean + 5 English samples through the Claude `count_tokens` API. The added English measurement lets us run the *asymmetry test*: compare Claude/GPT ratios on Korean vs English to tell apart "Korean-specific gap" from "general Claude inefficiency".

**Prereq**: `uv sync --extra anthropic --extra dev`, `ANTHROPIC_API_KEY` in env, **and a non-zero account balance** at console.anthropic.com.

**Cost**: `count_tokens` has zero per-call billing, but Anthropic's API requires a positive balance to access at all (otherwise you get HTTP 400 `"credit balance is too low"`). A one-time minimum top-up (e.g. $5) covers this entire project — Phase 1 token-counting will not draw it down meaningfully.

In [5]:
import os

anth = None
try:
    from korean_llm_cost.tokenizers.anthropic_tok import AnthropicTokenizer
except ImportError:
    print("anthropic SDK not installed. Run:  uv sync --extra anthropic --extra dev")
else:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not set.")
        print("  cp .env.example .env  →  fill in the key  →  restart kernel.")
    else:
        # Default to Sonnet 4.5 (stable). Override to claude-sonnet-4-7 once verified.
        anth = AnthropicTokenizer("claude-sonnet-4-5")
        print(anth, f"(chat overhead = {anth._overhead} tokens, calibrated)")

<AnthropicTokenizer claude-sonnet-4-5@2025-09> (chat overhead = 7 tokens, calibrated)


In [6]:
if anth is not None:
    # ~15 API calls total. Zero per-call billing.
    cla_ko_counts = [anth.count(text) for _, text in korean_samples]
    cla_en_counts = [anth.count(text)  for text in english_baseline]

    # ----- Korean per-sample table -----
    print("=== Korean (10 samples) ===")
    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4}  text")
    print("-" * 80)
    for (category, text), n_cla in zip(korean_samples, cla_ko_counts):
        n_chars = len(text)
        n_gpt = tok.count(text)
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {n_cla:>4}  {text}")

    # ----- English per-sample table -----
    print("\n=== English (5 samples) ===")
    print(f"{'chars':>5} {'gpt':>4} {'cla':>4}  text")
    print("-" * 80)
    for text, n_cla in zip(english_baseline, cla_en_counts):
        n_chars = len(text)
        n_gpt = tok.count(text)
        print(f"{n_chars:>5} {n_gpt:>4} {n_cla:>4}  {text}")

    # ----- Aggregate TPCs (4 numbers) -----
    gpt_ko_tpc = ko_tpc                      # from earlier cell
    gpt_en_tpc = en_tpc                      # from earlier cell
    cla_ko_tpc = sum(cla_ko_counts) / ko_chars
    cla_en_tpc = sum(cla_en_counts) / en_chars

    print("\n=== Aggregate TPC ===")
    print(f"  GPT-4o   Korean : {gpt_ko_tpc:.3f}")
    print(f"  GPT-4o   English: {gpt_en_tpc:.3f}")
    print(f"  Claude   Korean : {cla_ko_tpc:.3f}")
    print(f"  Claude   English: {cla_en_tpc:.3f}")

    # ----- Cross-model ratios (Claude / GPT-4o) -----
    ratio_ko = cla_ko_tpc / gpt_ko_tpc
    ratio_en = cla_en_tpc / gpt_en_tpc
    asymmetry = ratio_ko / ratio_en  # > 1 → Claude relatively worse on Korean

    print("\n=== Inter-model ratios (Claude / GPT-4o) ===")
    print(f"  Korean  ratio: {ratio_ko:.2f}×")
    print(f"  English ratio: {ratio_en:.2f}×")
    print(f"  Asymmetry (KO ratio / EN ratio): {asymmetry:.2f}×")

    # ----- One-line asymmetry verdict -----
    if asymmetry >= 1.3:
        verdict = "Korean-SPECIFIC gap — Claude penalty on Korean far exceeds its English penalty."
        impl = "Strong paper hook: tokenizer disparity is *language-specific*, not general."
    elif asymmetry <= 1.0 / 1.3:
        verdict = "Inverted asymmetry — Claude is RELATIVELY worse on English than Korean."
        impl = "Investigate before reporting; sample bias likely. Re-check on a balanced corpus."
    else:
        verdict = "General Claude inefficiency — Claude is uniformly costlier across both languages."
        impl = "Paper hook softer: 'Claude general tokenizer gap', not 'Korean-specific'."
    print(f"\n→ {verdict}")
    print(f"  {impl}")

    # ----- Per-provider sanity (each model has its own expected band) -----
    print("\n=== Sanity (per-provider expected ranges) ===")
    checks = [
        ("GPT-4o Korean ", gpt_ko_tpc, 0.50, 0.80),
        ("GPT-4o English", gpt_en_tpc, 0.15, 0.30),
        ("Claude Korean ", cla_ko_tpc, 0.90, 1.30),
        ("Claude English", cla_en_tpc, 0.20, 0.35),
    ]
    for label, val, lo, hi in checks:
        ok = lo <= val <= hi
        status = "PASS" if ok else f"FAIL ({val:.3f})"
        print(f"  {label} TPC in [{lo:.2f}, {hi:.2f}]?  {status}")

    # ----- Project signal threshold (1.3× on Korean ratio) -----
    print()
    if abs(ratio_ko - 1.0) >= 0.3:
        print(f"→ Korean ratio |{ratio_ko:.2f} − 1| ≥ 0.3 at n=10. Candidate H1/H2 signal.")
    else:
        print(f"→ Korean ratio |{ratio_ko:.2f} − 1| < 0.3 at n=10. No clear inter-model gap.")

=== Korean (10 samples) ===
category     chars  gpt  cla  text
--------------------------------------------------------------------------------
conv            12    8   17  오늘 점심 뭐 먹을까?
conv            15    9   18  내일 회의 시간 좀 알려줘.
news            23   13   24  한국은행은 기준금리를 동결하기로 결정했다.
news            23   13   24  정부는 내년도 예산안을 국회에 제출하였다.
medical         40   30   41  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22   17   26  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33   20   34  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38   22   35  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27   22   30  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11    8   13  와 진짜 대박이네요!

=== English (5 samples) ===
chars  gpt  cla  text
--------------------------------------------------------------------------------
   35    8    8  What should we eat for lunch today?
   38    8    8  Let me know the meeting time tomorrow.
   64   12   12  The Bank of Korea decided to freeze the benc

## Google (Gemini) check

Same 10 Korean + 5 English samples through the Gemini `count_tokens` API. Same calibration pattern (subtract any constant overhead measured from a 1-character probe).

**Prereq**: `uv sync --extra google --extra anthropic --extra dev`, `GOOGLE_API_KEY` (or `GEMINI_API_KEY`) in env.

**Cost**: Gemini's `count_tokens` runs against the Generative Language API **free tier**. No balance requirement, no per-call charge — get a key from https://aistudio.google.com/apikey and you're set. Free-tier rate limits are well above what Phase 1 needs.

**Expected outcomes** (working hypothesis from the Claude finding):
- If Gemini Korean TPC is roughly in `[0.65, 1.10]` and KPR is materially above GPT-4o's 3.39× → consistent with the *frontier-wide Korean-specific gap* hypothesis
- If Gemini Korean TPC ≈ 0.65 and KPR ≈ 3.4× (GPT-4o-like) → Anthropic-specific issue, not frontier-wide

In [7]:
gem = None
try:
    from korean_llm_cost.tokenizers.google_tok import GoogleTokenizer
except ImportError:
    print("google-generativeai not installed. Run:  uv sync --extra google --extra anthropic --extra dev")
else:
    if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
        print("GOOGLE_API_KEY not set.")
        print("  cp .env.example .env  →  fill in the key  →  restart kernel.")
        print("  (Get a free key at https://aistudio.google.com/apikey — no balance required.)")
    else:
        gem = GoogleTokenizer("gemini-2.5-flash")
        print(gem, f"(probe overhead = {gem._overhead} tokens, calibrated)")

/Users/krkarma777/Dev/paper/korean-llm-cost/src/korean_llm_cost/tokenizers/google_tok.py:26: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


<GoogleTokenizer gemini-2.5-flash@2025-06> (probe overhead = 0 tokens, calibrated)


In [8]:
if gem is not None:
    # ~15 API calls. Free tier, no charge.
    gem_ko_counts = [gem.count(text) for _, text in korean_samples]
    gem_en_counts = [gem.count(text)  for text in english_baseline]

    # ----- Korean per-sample table -----
    print("=== Korean (10 samples) ===")
    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4} {'gem':>4}  text")
    print("-" * 80)
    for i, (category, text) in enumerate(korean_samples):
        n_chars = len(text)
        n_gpt = tok.count(text)
        n_cla = cla_ko_counts[i] if anth is not None else None
        n_gem = gem_ko_counts[i]
        cla_str = f"{n_cla:>4}" if n_cla is not None else "   -"
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {cla_str} {n_gem:>4}  {text}")

    # ----- English per-sample table -----
    print("\n=== English (5 samples) ===")
    print(f"{'chars':>5} {'gpt':>4} {'cla':>4} {'gem':>4}  text")
    print("-" * 80)
    for i, text in enumerate(english_baseline):
        n_chars = len(text)
        n_gpt = tok.count(text)
        n_cla = cla_en_counts[i] if anth is not None else None
        n_gem = gem_en_counts[i]
        cla_str = f"{n_cla:>4}" if n_cla is not None else "   -"
        print(f"{n_chars:>5} {n_gpt:>4} {cla_str} {n_gem:>4}  {text}")

    # ----- Aggregate Gemini TPCs -----
    gem_ko_tpc = sum(gem_ko_counts) / ko_chars
    gem_en_tpc = sum(gem_en_counts) / en_chars
    gem_kpr = gem_ko_tpc / gem_en_tpc

    print("\n=== Gemini aggregate ===")
    print(f"  Gemini Korean : {gem_ko_tpc:.3f}")
    print(f"  Gemini English: {gem_en_tpc:.3f}")
    print(f"  Gemini KPR (ko/en): {gem_kpr:.2f}×")

    # ----- Per-Gemini sanity (loose initial range — tighten after observing) -----
    print()
    ok_gem_ko = 0.50 <= gem_ko_tpc <= 1.40
    ok_gem_en = 0.15 <= gem_en_tpc <= 0.40
    print("Gemini Korean  TPC in [0.50, 1.40]? ", "PASS" if ok_gem_ko else f"FAIL ({gem_ko_tpc:.3f})")
    print("Gemini English TPC in [0.15, 0.40]? ", "PASS" if ok_gem_en else f"FAIL ({gem_en_tpc:.3f})")
    print("  (Gemini ranges are loose — tighten in cell 0 after this run.)")

=== Korean (10 samples) ===
category     chars  gpt  cla  gem  text
--------------------------------------------------------------------------------
conv            12    8   17    8  오늘 점심 뭐 먹을까?
conv            15    9   18    9  내일 회의 시간 좀 알려줘.
news            23   13   24   13  한국은행은 기준금리를 동결하기로 결정했다.
news            23   13   24   15  정부는 내년도 예산안을 국회에 제출하였다.
medical         40   30   41   31  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22   17   26   16  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33   20   34   19  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38   22   35   19  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27   22   30   21  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11    8   13    7  와 진짜 대박이네요!

=== English (5 samples) ===
chars  gpt  cla  gem  text
--------------------------------------------------------------------------------
   35    8    8    8  What should we eat for lunch today?
   38    8    8    8  Let me know the meeting time t

## HuggingFace: Solar 10.7B (Upstage, Korean-specialized)

First HF wrapper. `AutoTokenizer.from_pretrained` downloads only the *tokenizer files* (~5–15 MB), never the model weights. Files are cached at `~/.cache/huggingface/hub/`, so the first run incurs a brief download and later runs are instant.

**Prereq**: `uv sync --extra hf --extra google --extra anthropic --extra dev`. No API key required (Solar is an open model on HF Hub).

**Working hypothesis being tested**: Korean-specialized models should show **KPR/GPT < 0.7** — i.e., Korean is *cheaper* per character than English, the opposite asymmetry from Claude. If confirmed, this is the H2 (Korean-specialized advantage) signal.

**Sanity range**: deliberately loose (`Korean: 0.30–0.80`, `English: 0.20–0.45`). Tighten in cell 0 after observing actual output.

In [9]:
solar = None
try:
    from korean_llm_cost.tokenizers.hf_tok import HFTokenizer
except ImportError:
    print("transformers not installed. Run:")
    print("  uv sync --extra hf --extra google --extra anthropic --extra dev")
else:
    print("Loading Solar tokenizer (first call downloads ~10MB to ~/.cache/huggingface/)...")
    solar = HFTokenizer("upstage/SOLAR-10.7B-Instruct-v1.0")
    print(solar, f"(probe overhead = {solar._overhead} tokens — expected 0 for HF)")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loading Solar tokenizer (first call downloads ~10MB to ~/.cache/huggingface/)...


<HFTokenizer upstage/SOLAR-10.7B-Instruct-v1.0@main> (probe overhead = 0 tokens — expected 0 for HF)


In [10]:
if solar is not None:
    sol_ko_counts = [solar.count(text) for _, text in korean_samples]
    sol_en_counts = [solar.count(text)  for text in english_baseline]

    # ----- Korean per-sample table (4 columns now: gpt / cla / gem / sol) -----
    print("=== Korean (10 samples) ===")
    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4} {'gem':>4} {'sol':>4}  text")
    print("-" * 80)
    for i, (category, text) in enumerate(korean_samples):
        n_chars = len(text)
        n_gpt = tok.count(text)
        n_cla = f"{cla_ko_counts[i]:>4}" if anth is not None else "   -"
        n_gem = f"{gem_ko_counts[i]:>4}" if gem is not None  else "   -"
        n_sol = sol_ko_counts[i]
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {n_cla} {n_gem} {n_sol:>4}  {text}")

    # ----- Aggregate Solar TPCs -----
    sol_ko_tpc = sum(sol_ko_counts) / ko_chars
    sol_en_tpc = sum(sol_en_counts) / en_chars
    sol_kpr = sol_ko_tpc / sol_en_tpc

    print("\n=== Solar aggregate ===")
    print(f"  Solar Korean : {sol_ko_tpc:.3f}")
    print(f"  Solar English: {sol_en_tpc:.3f}")
    print(f"  Solar KPR (ko/en): {sol_kpr:.2f}×")

    # ----- Per-Solar sanity (loose initial range) -----
    print()
    ok_sol_ko = 0.30 <= sol_ko_tpc <= 0.80
    ok_sol_en = 0.20 <= sol_en_tpc <= 0.45
    print("Solar Korean  TPC in [0.30, 0.80]? ", "PASS" if ok_sol_ko else f"FAIL ({sol_ko_tpc:.3f})")
    print("Solar English TPC in [0.20, 0.45]? ", "PASS" if ok_sol_en else f"FAIL ({sol_en_tpc:.3f})")
    print("  (Solar ranges are loose — tighten in cell 0 after this run.)")

=== Korean (10 samples) ===
category     chars  gpt  cla  gem  sol  text
--------------------------------------------------------------------------------
conv            12    8   17    8   19  오늘 점심 뭐 먹을까?
conv            15    9   18    9   20  내일 회의 시간 좀 알려줘.
news            23   13   24   13   24  한국은행은 기준금리를 동결하기로 결정했다.
news            23   13   24   15   24  정부는 내년도 예산안을 국회에 제출하였다.
medical         40   30   41   31   43  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22   17   26   16   31  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33   20   34   19   36  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38   22   35   19   37  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27   22   30   21   32  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11    8   13    7   12  와 진짜 대박이네요!

=== Solar aggregate ===
  Solar Korean : 1.139
  Solar English: 0.217
  Solar KPR (ko/en): 5.25×

Solar Korean  TPC in [0.30, 0.80]?  FAIL (1.139)
Solar English TPC in [0.20, 0.45]?  PASS
  (

## HuggingFace: EXAONE 3.5 7.8B (LG, Korean-specialized w/ extended tokenizer)

EXAONE is the *direct test* of the working hypothesis emerging from Solar:
**"Explicit Korean BPE coverage in the tokenizer dominates pretraining target language for cost efficiency."**

Solar (Mistral-derived, no tokenizer extension) showed KPR/GPT = 1.55× — Korean PENALTY despite Korean-targeted pretraining. EXAONE has its own tokenizer with explicit Korean vocabulary additions (Llama-style base + Korean tokens), so it directly tests whether tokenizer extension closes the gap.

**Two paper-relevant scenarios** (both lead to strong messages):
- **Scenario A — EXAONE KPR/GPT ≈ 1.0×** → confirms the hypothesis: explicit Korean BPE in the tokenizer is the deciding factor. Headline: *"Tokenizer Korean BPE coverage dominates pretraining language for cost."*
- **Scenario B — EXAONE also shows penalty** → hypothesis breaks; the gap is driven by something other than vocab extension (training corpus mix? merge ordering?). Headline pivots to investigating that.

**License note (paper Methods section requirement)**: EXAONE is released under the **EXAONE AI Model License Agreement**. Non-commercial academic research is permitted, but the paper's Methods section must cite the license and state the research purpose explicitly. Do not redistribute the tokenizer files; downloading via `transformers` cache is the supported path.

**Prereq**: same as Solar — `uv sync --extra hf --extra google --extra anthropic --extra dev`. EXAONE ships custom tokenizer code, so `trust_remote_code=True` is required.

In [11]:
exa = None
try:
    from korean_llm_cost.tokenizers.hf_tok import HFTokenizer
except ImportError:
    print("transformers not installed.")
    print("  uv sync --extra hf --extra google --extra anthropic --extra dev")
else:
    print("Loading EXAONE 3.5 tokenizer (~10MB to ~/.cache/huggingface/, first run only)...")
    print("  trust_remote_code=True is required — EXAONE ships custom tokenizer code.")
    print("  License: EXAONE AI Model License Agreement (non-commercial research OK,")
    print("           paper Methods section must cite the license).")
    exa = HFTokenizer(
        "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        trust_remote_code=True,
    )
    print(exa, f"(probe overhead = {exa._overhead} tokens — expected 0 for HF)")

Loading EXAONE 3.5 tokenizer (~10MB to ~/.cache/huggingface/, first run only)...
  trust_remote_code=True is required — EXAONE ships custom tokenizer code.
  License: EXAONE AI Model License Agreement (non-commercial research OK,
           paper Methods section must cite the license).


config.json: 0.00B [00:00, ?B/s]

configuration_exaone.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<HFTokenizer LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct@main> (probe overhead = 0 tokens — expected 0 for HF)


In [12]:
if exa is not None:
    exa_ko_counts = [exa.count(text) for _, text in korean_samples]
    exa_en_counts = [exa.count(text)  for text in english_baseline]

    # ----- Korean per-sample table — 5 model columns now -----
    print("=== Korean (10 samples) ===")
    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4} {'gem':>4} {'sol':>4} {'exa':>4}  text")
    print("-" * 90)
    for i, (category, text) in enumerate(korean_samples):
        n_chars = len(text)
        n_gpt = tok.count(text)
        n_cla = f"{cla_ko_counts[i]:>4}" if anth  is not None else "   -"
        n_gem = f"{gem_ko_counts[i]:>4}" if gem   is not None else "   -"
        n_sol = f"{sol_ko_counts[i]:>4}" if solar is not None else "   -"
        n_exa = exa_ko_counts[i]
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {n_cla} {n_gem} {n_sol} {n_exa:>4}  {text}")

    # ----- Aggregate EXAONE TPCs -----
    exa_ko_tpc = sum(exa_ko_counts) / ko_chars
    exa_en_tpc = sum(exa_en_counts) / en_chars
    exa_kpr = exa_ko_tpc / exa_en_tpc

    print("\n=== EXAONE aggregate ===")
    print(f"  EXAONE Korean : {exa_ko_tpc:.3f}")
    print(f"  EXAONE English: {exa_en_tpc:.3f}")
    print(f"  EXAONE KPR (ko/en): {exa_kpr:.2f}×")

    # ----- Per-EXAONE sanity (loose: hypothesis allows wide range) -----
    print()
    ok_exa_ko = 0.30 <= exa_ko_tpc <= 1.20
    ok_exa_en = 0.15 <= exa_en_tpc <= 0.45
    print("EXAONE Korean  TPC in [0.30, 1.20]? ", "PASS" if ok_exa_ko else f"FAIL ({exa_ko_tpc:.3f})")
    print("EXAONE English TPC in [0.15, 0.45]? ", "PASS" if ok_exa_en else f"FAIL ({exa_en_tpc:.3f})")
    print("  (Loose range — tighten in cell 0 once observed.)")

    # ----- Direct hypothesis check (the whole reason we measured EXAONE) -----
    baseline_kpr = ko_tpc / en_tpc
    exa_kpr_ratio = exa_kpr / baseline_kpr
    print(f"\n=== Hypothesis check: EXAONE KPR/GPT = {exa_kpr_ratio:.2f}× ===")
    if exa_kpr_ratio < 1.0 / 1.3:
        print("  → ADVANTAGE: EXAONE actually IMPROVES on GPT-4o for Korean.")
        print("    Strongest scenario — explicit Korean BPE training is decisive.")
    elif exa_kpr_ratio < 1.3:
        print("  → GPT-like cluster: EXAONE's Korean tokenizer is in the efficient group")
        print("    (GPT-4o, Gemini). Confirms working hypothesis at n=10.")
    else:
        print("  → PENALTY: EXAONE shows asymmetry like Claude/Solar despite tokenizer extension.")
        print("    Working hypothesis BREAKS. Investigate: corpus mix, merge order, vocab size?")

=== Korean (10 samples) ===
category     chars  gpt  cla  gem  sol  exa  text
------------------------------------------------------------------------------------------
conv            12    8   17    8   19    6  오늘 점심 뭐 먹을까?
conv            15    9   18    9   20    7  내일 회의 시간 좀 알려줘.
news            23   13   24   13   24   12  한국은행은 기준금리를 동결하기로 결정했다.
news            23   13   24   15   24   12  정부는 내년도 예산안을 국회에 제출하였다.
medical         40   30   41   31   43   29  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22   17   26   16   31   14  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33   20   34   19   36   16  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38   22   35   19   37   18  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27   22   30   21   32   18  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11    8   13    7   12    6  와 진짜 대박이네요!

=== EXAONE aggregate ===
  EXAONE Korean : 0.566
  EXAONE English: 0.200
  EXAONE KPR (ko/en): 2.83×

EXAONE Korean  TPC in [0

In [13]:
# Unified summary across every provider successfully measured in this notebook.
# Each row: per-model TPC + per-model KPR + KO ratio relative to GPT-4o baseline
# + KPR/GPT (the language-specific asymmetry signal).

rows: list[tuple[str, float, float]] = [("GPT-4o", ko_tpc, en_tpc)]
if anth is not None:
    rows.append(("Claude Sonnet 4.5", cla_ko_tpc, cla_en_tpc))
if gem is not None:
    rows.append(("Gemini 2.5 Flash", gem_ko_tpc, gem_en_tpc))
if solar is not None:
    rows.append(("Solar 10.7B", sol_ko_tpc, sol_en_tpc))
if exa is not None:
    rows.append(("EXAONE 3.5 7.8B", exa_ko_tpc, exa_en_tpc))

baseline_ko = ko_tpc
baseline_kpr = ko_tpc / en_tpc

print(f"{'Model':<22} {'KO':>6} {'EN':>6} {'KPR':>7} {'KO/GPT':>8} {'KPR/GPT':>9}")
print("-" * 70)
for name, ko, en in rows:
    kpr = ko / en
    ko_ratio = ko / baseline_ko
    kpr_ratio = kpr / baseline_kpr
    print(f"{name:<22} {ko:>6.3f} {en:>6.3f} {kpr:>6.2f}× {ko_ratio:>7.2f}× {kpr_ratio:>8.2f}×")

# ----- Per-model verdict (auto-categorize each model's KPR/GPT) -----
print("\n=== Per-model verdict (Korean-specific asymmetry vs GPT-4o) ===")
for name, ko, en in rows:
    kpr_ratio = (ko / en) / baseline_kpr
    if name == "GPT-4o":
        verdict = "baseline"
    elif kpr_ratio >= 1.3:
        verdict = "Korean-specific PENALTY (≥1.3× asymmetry)"
    elif kpr_ratio <= 1.0 / 1.3:
        verdict = "Korean-specific ADVANTAGE (≤0.77× asymmetry)"
    else:
        verdict = "GPT-4o-like (no language-specific asymmetry)"
    print(f"  {name:<22} KPR/GPT = {kpr_ratio:.2f}×  → {verdict}")

# ----- Aggregate paper-relevant counts -----
non_baseline = [(n, (k/e)/baseline_kpr) for n, k, e in rows if n != "GPT-4o"]
n_penalty   = sum(1 for _, r in non_baseline if r >= 1.3)
n_advantage = sum(1 for _, r in non_baseline if r <= 1.0/1.3)
n_neutral   = len(non_baseline) - n_penalty - n_advantage

print("\n=== Paper-relevant counts (n=10/n=5 — informational only) ===")
print(f"  Korean-specific PENALTY  : {n_penalty} model(s)")
print(f"  GPT-4o-like              : {n_neutral} model(s)")
print(f"  Korean-specific ADVANTAGE: {n_advantage} model(s)")
if n_penalty >= 1 and n_advantage >= 1:
    print("\n  → Two contrasting categories observed (penalty + advantage). ")
    print("    Paper table 1 form: Korean-specialized vs Frontier vs Anthropic-outlier.")
    print("    Both H1 (frontier variance) and H2 (Korean-specialized win) signals present.")
elif n_penalty >= 2 and n_advantage == 0:
    print("\n  → Two-cluster pattern (penalty cluster vs GPT-like cluster). ")
    print("    Working hypothesis (n=10): tokenizer Korean BPE coverage *appears to*")
    print("    dominate pretraining target language. Verify on full Phase 1 corpus.")

print("\n(All numbers above are n=10/n=5 — do not paper-cite. Re-measure on full corpus.)")

Model                      KO     EN     KPR   KO/GPT   KPR/GPT
----------------------------------------------------------------------
GPT-4o                  0.664  0.196   3.39×    1.00×     1.00×
Claude Sonnet 4.5       1.074  0.213   5.05×    1.62×     1.49×
Gemini 2.5 Flash        0.648  0.200   3.24×    0.98×     0.95×
Solar 10.7B             1.139  0.217   5.25×    1.72×     1.55×
EXAONE 3.5 7.8B         0.566  0.200   2.83×    0.85×     0.83×

=== Per-model verdict (Korean-specific asymmetry vs GPT-4o) ===
  GPT-4o                 KPR/GPT = 1.00×  → baseline
  Claude Sonnet 4.5      KPR/GPT = 1.49×  → Korean-specific PENALTY (≥1.3× asymmetry)
  Gemini 2.5 Flash       KPR/GPT = 0.95×  → GPT-4o-like (no language-specific asymmetry)
  Solar 10.7B            KPR/GPT = 1.55×  → Korean-specific PENALTY (≥1.3× asymmetry)
  EXAONE 3.5 7.8B        KPR/GPT = 0.83×  → GPT-4o-like (no language-specific asymmetry)

=== Paper-relevant counts (n=10/n=5 — informational only) ===
  Korean-speci

## What to do if a check fails

Each provider has its own TPC band — a Claude Korean TPC of 1.07 is **expected behavior**, not a wrapper bug. Only a value far outside the provider's band is a real failure.

- **Korean TPC outside the provider's band by > 30%**: usually encoding (UTF-8 vs NFD), wrong model name, or `count` being called on bytes. Print `tok.info` and confirm the snapshot.
- **Calibration overhead negative** (any provider): the 1-character probe assumption (`'.' = 1 token`) may have broken in a future model release. Re-derive overhead by sending two probes of known different lengths and solving for the constant.
- **Claude HTTP 400 "credit balance is too low"**: API key works but balance is empty. Top up at console.anthropic.com. `count_tokens` itself is free.
- **Gemini HTTP 429 / quota errors**: free-tier rate limit hit. Either wait a minute or split the run into smaller batches. No paid tier required for Phase 1.
- **Gemini ImportError on `google.generativeai`**: pyproject's old SDK name. Run `uv sync --extra google --extra anthropic --extra dev` to install.
- **HF model gated / 401 on download**: model requires accepting a license + HF token. Llama family needs `HF_TOKEN`. Solar, Qwen, Gemma are open. EXAONE is open but uses a model-specific license (see below).
- **English TPC < 0.15**: extremely common-word content; the 5-sentence sanity sample is unrepresentatively easy. Not blocking — verify on the news/academic corpus before reporting.

## Model licenses to cite in the paper Methods section

- **GPT-4o, Claude, Gemini**: API access; cite the model snapshot date and SDK version, no license redistribution concern.
- **Solar (Upstage)**: Apache 2.0 (model and tokenizer). Permissive.
- **EXAONE 3.5**: **EXAONE AI Model License Agreement** (model-specific). Non-commercial academic research is permitted; the paper's Methods section must cite the license and state the research purpose explicitly. Do not redistribute tokenizer files; downloading via `transformers` cache is the supported path.
- **Llama 3.x (later)**: Llama 3 Community License Agreement; requires accepting on HuggingFace. Cite the version.
- **Qwen 2.5 (later)**: Apache 2.0 / Tongyi Qianwen License (varies by variant). Check per-model card.

## Next

Once GPT-4o + Claude + Gemini + Solar + EXAONE all PASS their bands, the next wrappers are Llama 3.1 and Qwen 2.5 — same HF pattern. Then Phase 1 wrapper inventory is complete and we move on to `metrics.py` and corpus collection.

## Early signals (n=10 / n=5, **NOT** for paper)

The sanity sample is too small for any number to appear in the paper. The signals below are **candidates** to verify on the full Phase 1 corpus (≥ 1000 sentences/category, bootstrap CI, paired tests).

### Within-model (GPT-4o, n=10)

| Candidate signal | Observed | Verify on |
|---|---|---|
| Korean ≈ 3.4× English per char (KPR) | 3.39 | parallel ko↔en corpus (KorNLI/MNLI overlap, OpenSubtitles ko-en) |
| Hanja-mixed text exceeds 0.8 TPC | 0.815 | hanja-rich subcorpus (legal / historical text) |
| Medical 13–17% above conversational | 0.75–0.77 vs 0.60–0.67 | KorMedMCQA + AI Hub medical text |
| Code-mixed *lower* than expected | 0.579 (English identifiers like `useState` likely 1 token) | dev-blog crawl (Velog/Tistory) |

### Cross-model (frontier comparison)

| Candidate signal | Observed | Verify on |
|---|---|---|
| Claude is the lone frontier outlier (Korean) | KO ratio 1.62×, EN ratio 1.09×, KPR/GPT = **1.49×** | full Phase 1 corpus, 6 models × 3 categories |
| Gemini matches GPT-4o on Korean | KO ratio 0.98×, KPR/GPT = 0.95× | parallel ko↔en corpus |
| Code-mixed: Gemini *more* efficient than GPT | 19 vs 22 tokens on `useState` snippet | Velog/Tistory dev-blog crawl |
| Claude English parity with GPT on simple text | 3/5 English samples gave *identical* token counts | larger English baseline |
| **REJECTED hypothesis**: "frontier-wide Korean gap" | only Claude shows KPR/GPT ≥ 1.3× | n/a (settled) |

### Cross-model (Korean-specialized, n=10/n=5)

| Candidate signal | Observed | Verify on |
|---|---|---|
| **Solar KPR/GPT** (H2 1차 검증) | _filled in by unified summary cell_ | KorMedMCQA + KLUE + parallel corpus |
| Working hypothesis: Solar KPR/GPT < 0.7 | **PENDING** | first Solar run |

**Interpretation rules** (so we read each new model's output the same way):
- **A model's KPR/GPT ≥ 1.3×** → Korean-specific PENALTY vs GPT baseline
- **A model's KPR/GPT between 1/1.3× and 1.3×** → GPT-4o-like (no language-specific asymmetry)
- **A model's KPR/GPT ≤ 1/1.3×** → Korean-specific ADVANTAGE (Korean cheaper than English vs GPT)

**Paper main hook (격상 2026-05-07):**
*"Of three frontier tokenizers tested, two (GPT-4o, Gemini 2.5 Flash) handle Korean comparably; Claude alone exhibits a 1.49× language-specific asymmetry — equivalent to GPT on English (1.09× ratio) but 1.62× more expensive on Korean."* If Solar/EXAONE confirm KPR/GPT < 0.7 (H2), the paper table splits cleanly into three categories: *frontier-average / Anthropic-outlier / Korean-specialized-advantage*.

**Project signal threshold (decision rule, 2026-05-07):**
- Inter-model TPC ratio **≥ 1.3×** at full-corpus scale → H1/H2 candidate signal, primary paper angle
- Inter-model TPC ratio **< 1.3×** → pivot toward domain-variance (H3) as the primary angle

**Do not cite n=10 numbers anywhere outside this notebook.** They exist only to motivate which directions are worth prioritizing in the full Phase 1 run.